In [ ]:
import pandas as pd
import numpy as np

data_dir = r'../data/'

with open(data_dir + 'MPCORB.DAT', "r") as f:
  for i, line in enumerate(f):
    if not line.startswith("00001"):
      continue
    header_rows = i
    break
    
# Columns
colspecs = [
  (0, 7),      # packed_designation
  (8, 13),     # H
  (14, 19),    # G
  (20, 25),    # epoch_packed
  (26, 35),    # M
  (37, 46),    # arg_peri
  (48, 57),    # node
  (59, 68),    # inclination
  (70, 79),    # eccentricity
  (80, 91),    # mean_motion
  (92, 103),   # semimajor_axis
  (105, 106),  # U
  (107, 116),  # reference
  (117, 122),  # n_obs
  (123, 126),  # n_opp
  (127, 136),  # arc_or_years
  (137, 141),  # rms
  (142, 145),  # pert_coarse
  (146, 149),  # pert_precise
  (150, 160),  # computer
  (161, 165),  # flags
  (166, 194),  # readable_designation
  (194, 202),  # last_observation
]

# Names of columns
names = [
  "packed_designation",
  "H",
  "G",
  "epoch_packed",
  "mean_anomaly",
  "arg_perihelion",
  "ascending_node",
  "inclination",
  "eccentricity",
  "mean_motion",
  "semimajor_axis",
  "U",
  "reference",
  "n_observations",
  "n_oppositions",
  "arc_or_years",
  "rms",
  "perturbers_coarse",
  "perturbers_precise",
  "computer",
  "flags",
  "designation",
  "last_observation"
]

# Dataframe conversion
df = pd.read_fwf(
    data_dir + 'MPCORB.DAT',
    colspecs=colspecs,
    names=names,
    skiprows=header_rows
)

# Classify as numbered or provisional (state)
df['is_numbered'] = df['designation'].str.startswith("(")
df['is_provisional'] = ~df['is_numbered']

# Classify the orbit uncertainty (U)
df['orbit_uncertainty'] = (pd.to_numeric(df['U'], errors='coerce'))
df['is_eccentricity_assumed'] = df['U'].isin(['E', 'F'])
df['has_multiple_designation'] = df['U'].isin(['D', 'F'])

# Compute perihelion distance (q = a(1-e)) [AU]
# https://markmcintyreastro.co.uk/wp-content/uploads/sites/3/2025/07/HowTo-Computing-planetary-positions.pdf
df['perihelion_distance'] = df['semimajor_axis'] * (1 - df['eccentricity'])

# Classify as NEOs (q < 1.3 AU)
# https://d1wqtxts1xzle7.cloudfront.net/66515364/3048-libre.pdf?1619161639=&response-content-disposition=inline%3B+filename%3DPhysical_properties_of_near_Earth_object.pdf&Expires=1782916101&Signature=IK0caz2YzJ44H42WeWqENPnV99jUNhuY2LacKnsyJ1ferA1jyA8jfo2fkWAie3lnGtRCYrXmdG5tfkwSsI132Gq40QM4GPcedmydv8zB3AKlMihtfoxVvfaqitPcmHPkx~U0eeMkqWMy43Gnbyxw2RhIndF3sAuMVli0~nu1uY7eSQS1VkCqkf4rp91Not2xJ6-XoHoUN9OJmnDUEfKHRXbJYgdQlGlZziQgOfy5tsUMsjzlRrbcqTZA~OH~d~x5TRu6ed6cv56sxdEzHhCBOvlBzYPoNcmJrlzb58wWPrWXUA7d1QoZYagZczkAQoWHwSpbgZCIEAaJIVv603-~Ow__&Key-Pair-Id=APKAJLOHF5GGSLRBV4ZA
# https://iopscience.iop.org/article/10.3847/1538-3881/ae2fc5/pdf
# https://link.springer.com/content/pdf/10.1007/s00159-013-0065-4.pdf
# Also SBDB definition is q < 1.3 
df['is_neo'] = df['perihelion_distance'] < 1.3

# Classify as MCA (1.3 < q < 1.666 && a < 3.2)
mca_conditions = [
  (df['semimajor_axis'] < 3.2) &
  (df['perihelion_distance'] > 1.3) &
  (df['perihelion_distance'] < 1.666)
]
df['is_mca'] = np.select(mca_conditions, [True], default=False)

# Classify as MBAs (2.0 AU < a < 3.2 AU; e < 0.3; q > 1.3 AU)
# https://arxiv.org/pdf/astro-ph/9801023
# https://iopscience.iop.org/article/10.1086/429734/pdf
# https://www.aanda.org/articles/aa/pdf/2017/02/aa29252-16.pdf
# Also SBD: 1.666 < q && 2 < a < 4.6
mba_conditions = [
  (df['semimajor_axis'] > 2.0) & 
  (df['semimajor_axis'] < 4.6) &
  (df['perihelion_distance'] > 1.666)
]
df['is_mba'] = np.select(mba_conditions, [True], default=False)

# Classify Jupiter Trojans (4.95 AU < a < 5.45 AU; e < 0.6; I < 40°)
# https://www.aanda.org/articles/aa/pdf/2019/11/aa36600-19.pdf
# https://iopscience.iop.org/article/10.3847/1538-3881/ad2200/pdf
# SBDB: 4.6 < a < 5.5 && e < 0.3
jup_conditions = [
  (df['semimajor_axis'] > 4.6) &
  (df['semimajor_axis'] < 5.5) &
  (df['eccentricity'] < 0.3)
]
df['is_tjn'] = np.select(jup_conditions, [True], default=False)

# Classify centaurs (5.2 AU < a < 30 AU; I < 80°; not jupiter trojan)
# https://arxiv.org/pdf/2506.04483
# https://iopscience.iop.org/article/10.1088/0004-6256/137/5/4296/pdf
# SBDB: 5.5 < a < 30.1
centaur_conditions = [
  (df['semimajor_axis'] > 5.5) &
  (df['semimajor_axis'] < 30.1) &
  (~df['is_tjn'])
]
df['is_cen'] = np.select(centaur_conditions, [True], default=False)

# Classify TNOs (I consider classic so far: q > 1.3 AU; 41 AU < a; e < 0.25; I < 32°)
# https://link.springer.com/content/pdf/10.1007/s001590100014.pdf
# https://www.aanda.org/articles/aa/pdf/2013/06/aa21090-13.pdf
# SBDB: 30.1 < a
tno_conditions = [
  (df['semimajor_axis'] > 30.1)
]
df['is_tno'] = np.select(tno_conditions, [True], default=False)

# PAA (e = 1)
df['is_paa'] = df['eccentricity'] == 1

# HYA (e > 1)
df['is_hya'] = df['eccentricity'] > 1

# AST (others)
conditions = [
  (~df['is_neo']) &
  (~df['is_mca']) &
  (~df['is_mba']) &
  (~df['is_tjn']) &
  (~df['is_cen']) &
  (~df['is_tno']) &
  (~df['is_paa']) &
  (~df['is_hya'])
]
df['is_ast'] = np.select(conditions, [True], default=False)

# Separate classification dataframe
df_classification = df[[
  'designation', 
  'is_numbered', 
  'is_provisional', 
  'orbit_uncertainty',
  'is_eccentricity_assumed',
  'has_multiple_designation',
  'is_neo',
  'is_mca',
  'is_mba',
  'is_tjn',
  'is_cen',
  'is_tno',
  'is_paa',
  'is_hya',
  'is_ast'
]]

# Handle numbered + designation combinations
df_classification['designation'] = df_classification['designation'].str.replace(r"\(\d+\)\s(.*)", r"\1", regex=True)

# Write new dataframe to a csv
df_classification.to_csv(data_dir + 'body_classification.csv', index=False)